# Raio-X do Mercado de Trabalho Industrial de SC — pipeline

Datathon CDIA 2026 · UniSENAI. O contrato do projeto é `docs/SDD_v2.md`.

**Pergunta:** quais regiões e setores industriais de SC enfrentam apagão de mão de obra ou expansão acelerada, e como renda e escolaridade explicam esses gargalos?

**Tese testada:** "retenção, não escassez". Este notebook só chama as funções de `src/` e explica cada etapa; ele roda de cima a baixo (`Restart & Run All`).

In [1]:
import sys, os
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))
import polars as pl
import config as cfg
from src import ingest, referencias, clean, macrofamilias, kpis, testes, figuras, recomendacoes
from src.log import iniciar_log
pl.Config.set_tbl_rows(20); pl.Config.set_tbl_width_chars(160)
# Sem internet (ou para não rebaixar): DATATHON_OFFLINE=1
OFFLINE = cfg.MUNICIPIOS_REGIOES_CSV.exists() or os.getenv('DATATHON_OFFLINE') == '1'

## 1. Ingestão (R1–R2)

O CSV de 413 MB é lido uma vez com Polars, com **todas as colunas como texto** (códigos nunca viram inteiro), renomeado para ASCII e salvo em `data/interim/caged_raw.parquet`.

In [2]:
iniciar_log()
raw = ingest.main()
raw.head()


## Ingestão (R1–R2)

| Item | Valor |
| --- | --- |
| Linhas lidas | 3,397,416 |
| Colunas | 28 |
| Nulos em idade | 2,202 |
| Salário não numérico | 0 |
| Tempo de leitura | 3.5 s |
| Pico de memória do processo | 1,927 MB |



competenciamov,regiao,uf,municipio,secao,subclasse,saldomovimentacao,cbo2002ocupacao,categoria,graudeinstrucao,idade,horascontratuais,racacor,sexo,tipoempregador,tipoestabelecimento,tipomovimentacao,tipodedeficiencia,indtrabintermitente,indtrabparcial,salario,tamestabjan,indicadoraprendiz,origemdainformacao,competenciadec,indicadordeforadoprazo,unidadesalariocodigo,valorsalariofixo
str,str,str,str,str,str,i8,str,str,str,i16,f64,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,f64
"""202509""","""4""","""42""","""420890""","""F""","""4120400""",1,"""715505""","""101""","""7""",40,44.0,"""1""","""1""","""0""","""1""","""97""","""0""","""0""","""0""",2018.0,"""5""","""0""","""1""","""202509""","""0""","""5""",2018.0
"""202509""","""4""","""42""","""421605""","""G""","""4671100""",-1,"""632120""","""101""","""7""",47,44.0,"""1""","""1""","""0""","""1""","""40""","""0""","""0""","""0""",1693.35,"""5""","""0""","""1""","""202509""","""0""","""5""",1693.35
"""202509""","""4""","""42""","""420820""","""F""","""4120400""",1,"""715505""","""101""","""6""",30,44.0,"""1""","""1""","""0""","""1""","""97""","""0""","""0""","""0""",2556.0,"""5""","""0""","""1""","""202509""","""0""","""5""",2556.0
"""202509""","""4""","""42""","""420820""","""G""","""4530701""",-1,"""411010""","""101""","""9""",33,44.0,"""3""","""3""","""0""","""1""","""31""","""0""","""0""","""0""",3181.34,"""7""","""0""","""1""","""202509""","""0""","""5""",3181.34
"""202509""","""4""","""42""","""421555""","""C""","""2512800""",-1,"""253405""","""101""","""8""",34,34.0,"""1""","""3""","""0""","""1""","""40""","""0""","""0""","""0""",1650.0,"""4""","""0""","""1""","""202509""","""0""","""5""",1650.0


## 2. Referências (R11, R12, R12b)

Municípios do evento (sem o rodapé HTML), títulos de CBO e subclasses do dicionário. As microrregiões e o GeoJSON vêm da API do IBGE e precisam de internet; se já existirem em `data/reference/`, não são baixados de novo.

In [3]:
referencias.main(offline=OFFLINE)
SEM_REGIOES = not cfg.MUNICIPIOS_REGIOES_CSV.exists()
print('Regiões do IBGE disponíveis:', not SEM_REGIOES)


## Dicionário (R12b)

| Item | Valor |
| --- | --- |
| Títulos de CBO | 2,778 |
| Subclasses CNAE | 1,357 |


## Municípios do evento (R11)

| Item | Valor |
| --- | --- |
| Municípios válidos | 295 |
| População total (Censo 2022) | 7,610,361 |

Modo offline: regiões e GeoJSON do IBGE não foram baixados.
Regiões do IBGE disponíveis: False


## 3. Limpeza, filtros e junções (R3–R14)

Janela set/2025–ago/2026, seções B–F, base de salário, faixas de escolaridade, grupos setoriais e regiões. Cada regra registra no log quantas linhas entraram e saíram.

In [4]:
ind = clean.main(sem_regioes=SEM_REGIOES)
print(open(cfg.LOG_PIPELINE, encoding='utf-8').read()[-2500:])


## Limpeza e junções (R3–R14)

| Item | Valor |
| --- | --- |
| Entrada (caged_raw.parquet) | 3,397,416 |
| R4 janela 202509–202608 | 3,397,416 → 3,391,288 (−6,128) |
| R5 seções BCDEF | 3,391,288 → 1,139,633 (−2,251,655) |
| R6 exclusões | coluna ausente: nada removido (V2) |
| R7 cópias excedentes de linhas idênticas mantidas (B–F) | 26,916 |
| R7 fora do prazo mantidas | 9,599 |
| R11 municípios casados | 295 de 295; sem população: 0 |
| R12 regiões | NÃO APLICADO (--sem-regioes). Rodar src/referencias.py com internet. |
| R12b CBO sem título | 131 linhas |
| R12b subclasse sem descrição | 0 linhas |
| Nulos em divisao_cnae | 0 |
| Nulos em cbo2002ocupacao | 0 |
| Nulos em grupo_setorial | 0 |
| Nulos em microrregiao | 1,139,633 |
| Saída (caged_ind.parquet) | 1,139,633 linhas, 41 colunas |

# Log do pipeline

Execução: 2026-11-10 15:17

## Ingestão (R1–R2)

| Item | Valor |
| --- | --- |
| Linhas lidas | 3,397,416 |
| Colunas | 28 |
| Nulos em idade | 2,202 |
| Salário não numéric

## 4. Teste T1 e conferência dos valores de referência

O saldo jan–mai/2026 é comparado com o boletim de emprego industrial da FIESC. Na construção a diferença é de 54 vagas (0,504%), pouco acima da tolerância de 0,5%; a equipe decidiu declarar a diferença (V19).

In [5]:
del raw
cfg.TESTES_MD.unlink(missing_ok=True)  # recomeça o registro dos testes
testes.t1_reconciliacao(ind)

| 2026-11-10 | T1 | Transformação (C) | 24,271 | 24,269 | diferença ≤ 0,5% | aprovado (0.01%) |
| 2026-11-10 | T1 | Indústria geral (B–E) | 24,536 | 24,527 | diferença ≤ 0,5% | aprovado (0.04%) |
| 2026-11-10 | T1 | Construção (F) | 10,775 | 10,721 | diferença ≤ 0,5% | REPROVADO (0.50%) |
| 2026-11-10 | T1 | Indústria total (B–F) | 35,311 | 35,248 | diferença ≤ 0,5% | aprovado (0.18%) |


False

## 5. Macrofamílias de CBO com IA generativa (Gemini)

O LLM classifica as ~354 CBOs que cobrem 95% dos movimentos em **área × nível de formação**, no formato do Mapa do Trabalho Industrial (SENAI/CNI), escolhendo só valores de uma lista fechada (enums no esquema de saída). Por padrão (`RODAR_LLM = False`) o notebook lê `data/reference/macrofamilias.csv` e não chama a API. O prompt completo:

In [6]:
print(macrofamilias.texto_prompt())

Você é um especialista em classificação de ocupações (CBO 2002) e em formação profissional industrial
no Brasil, no formato do Mapa do Trabalho Industrial (SENAI/CNI).

Classifique cada ocupação da lista em UMA área e UM nível de formação, usando SOMENTE os valores abaixo.
Nunca crie categorias, nunca altere nem invente códigos de CBO. Devolva exatamente as CBOs recebidas, uma vez cada.

ÁREAS:
- Alimentos e bebidas: Ocupações específicas do processamento de alimentos e bebidas: abate, desossa, laticínios, panificação industrial
- Têxtil, confecção e calçados: Fiação, tecelagem, corte, costura, acabamento, calçados
- Metalmecânica e soldagem: Usinagem, caldeiraria, soldagem, fundição, estamparia, montagem mecânica especializada
- Manutenção industrial: Mecânicos, eletricistas e eletromecânicos de manutenção, lubrificadores
- Eletroeletrônica e automação: Montagem e operação eletroeletrônica, instrumentação, automação, bobinagem
- Madeira e móveis: Serraria, marcenaria, montagem de móve

In [7]:
macrofamilias.main()
if cfg.MACROFAMILIAS_CSV.exists():
    display(pl.read_csv(cfg.MACROFAMILIAS_CSV, infer_schema_length=0).head(20))

CBOs de B–F: 1578 · no escopo (95% dos movimentos): 354 · sem título no dicionário: 9 (0 no escopo)
Nada a classificar ainda: sem chave/cache e sem resposta do plano B.
  Caminho A: crie .env com GEMINI_API_KEY e rode `python src/macrofamilias.py --llm`.
  Caminho B: siga prompts/plano_b_chat.md.


Código da chamada (resumo de `src/macrofamilias.py`):

```python
client = genai.Client()  # GEMINI_API_KEY vem do .env
config = types.GenerateContentConfig(system_instruction=texto_prompt(), temperature=0,
    response_mime_type='application/json', response_schema=list[Classificacao])
resp = client.models.generate_content(model=cfg.GEMINI_MODEL, contents=linhas_entrada(lote), config=config)
```

## 6. KPIs (seção 5)

1. **% de saídas a pedido** (adaptação da taxa de *quits* do JOLTS/BLS) · 2. **saldo por 10 mil habitantes** · 3. **prêmio de contratação** e 4. **descasamento de escolaridade**, ambos calculados dentro da mesma CBO e microrregião e ponderados (5.2).

In [8]:
saidas = kpis.main()
pl.read_csv(cfg.PROCESSED / 'kpis_grupo_setorial.csv').filter(pl.col('segmento') == 'Transformação').select('grupo_setorial', 'movimentos', 'saldo', 'kpi1_pct_a_pedido', 'kpi2_saldo_10mil_hab').sort('kpi1_pct_a_pedido', descending=True)


## KPIs (seção 5)

| Item | Valor |
| --- | --- |
| kpis_segmento | 3 linhas |
| kpis_grupo_setorial | 13 linhas |
| kpis_divisao | 37 linhas |
| kpis_segmento_sem_cbo_dominante | 3 linhas |
| kpis_grupo_setorial_sem_cbo_dominante | 13 linhas |
| perfil_escolaridade_grupo | 104 linhas |
| perfil_salario_grupo | 26 linhas |
| Recortes por microrregião, quadrantes e QL | PENDENTES: falta municipios_regioes.csv (V21) |
| Recortes por macrofamília | PENDENTES: falta macrofamilias.csv (Fase 2) |
| Tamanho de data/processed | 0.02 MB (limite 20 MB) |



grupo_setorial,movimentos,saldo,kpi1_pct_a_pedido,kpi2_saldo_10mil_hab
str,i64,i64,f64,f64
"""Alimentos e bebidas""",182711,4731,56.188336,6.216525
"""Madeira e móveis""",78864,-2710,53.141932,-3.560935
"""Têxtil, confecção e calçados""",195178,-5916,52.397386,-7.773613
"""Eletroeletrônica""",42338,1646,50.122874,2.162841
"""Plástico e borracha""",65547,265,49.673723,0.34821
"""Papel, celulose e gráfica""",27488,340,48.320318,0.446759
"""Minerais não metálicos e cerâm…",33492,-842,47.375779,-1.106386
"""Química e farmacêutica""",16391,341,46.105919,0.448073
"""Metalmecânica""",178307,3459,45.616764,4.545119


## 7. Testes T2 a T5

- **T2** separa efeito de setor de efeito da mistura de ocupações (padronização indireta).
- **T3** mede quanto o resultado depende do alimentador de linha (CBO 784205).
- **T4** checa estabilidade entre semestres. **T5** checa se a tese depende da definição de indústria.

In [9]:
testes.main_fase3()

| 2026-11-10 | T2a | C, mesma CBO em ≥3 divisões | 15 CBOs; amplitude mediana 11.1 p.p.; 15 acima de 5 p.p. | ≥ 10 CBOs | informativo (reportar se < 10 CBOs) | aprovado |


| 2026-11-10 | T2b | C, 14 maiores divisões | amplitude padronizada 5.6 p.p. (observada 13.3) | > 5 p.p. | diferença entre maior e menor taxa padronizada | aprovado |
shape: (14, 5)
┌──────────────┬───────────┬───────────────────────┬─────────────┬─────────────────────────────────┐
│ divisao_cnae ┆ observado ┆ esperado_pela_mistura ┆ padronizado ┆ grupo_setorial                  │
│ ---          ┆ ---       ┆ ---                   ┆ ---         ┆ ---                             │
│ str          ┆ f64       ┆ f64                   ┆ f64         ┆ str                             │
╞══════════════╪═══════════╪═══════════════════════╪═════════════╪═════════════════════════════════╡
│ 29           ┆ 52.7      ┆ 50.1                  ┆ 53.5        ┆ Metalmecânica                   │
│ 16           ┆ 54.2      ┆ 51.9                  ┆ 53.3        ┆ Madeira e móveis                │
│ 10           ┆ 56.5      ┆ 54.6                  ┆ 52.9        ┆ Alimentos e bebidas             │
│ 27      

| 2026-11-10 | T3 | KPI 1, 5 piores divisões sem CBO 784205 | ordem com: 10, 14, 16, 21, 27; maior mudança 4 | ≤ 1 posição | ranking estável | REPROVADO |


| 2026-11-10 | T3 | KPI 2, 5 piores divisões sem CBO 784205 | ordem com: 14, 16, 31, 23, 13; maior mudança 3 | ≤ 1 posição | ranking estável | REPROVADO |
| 2026-11-10 | T4 | C, município × grupo setorial (provisório, sem regiões) | r = 0.86 em 153 recortes | > 0,7 | estabilidade entre semestres | pendente |


| 2026-11-10 | T5 | KPI 1 do estado, C × B–E | 50.91% × 50.63% (dif. 0.28 p.p.) | < 1 p.p. |  | aprovado |
| 2026-11-10 | T5 | Top 5 microrregiões por KPI 1, C × B–E | aguardando regiões do IBGE (V21) | ≤ 1 troca |  | pendente |
| 2026-11-10 | T6 | Amostra M3 | planilha ainda não gerada (Fase 2) | ≥ 85% | concordância LLM × revisores | pendente |


In [10]:
pl.read_csv('reports/testes_detalhe/t2b_padronizacao_14_divisoes.csv', schema_overrides={'divisao_cnae': pl.Utf8})

divisao_cnae,observado,esperado_pela_mistura,padronizado,grupo_setorial
str,f64,f64,f64,str
"""29""",52.676123,50.056566,53.533125,"""Metalmecânica"""
"""16""",54.221553,51.877186,53.257936,"""Madeira e móveis"""
"""10""",56.53624,54.554966,52.894842,"""Alimentos e bebidas"""
"""27""",52.901685,51.42589,52.389363,"""Eletroeletrônica"""
"""14""",55.693225,54.325935,52.280858,"""Têxtil, confecção e calçados"""
"""31""",51.611947,50.405149,52.120367,"""Madeira e móveis"""
"""33""",44.161293,44.006895,51.067966,"""Metalmecânica"""
"""17""",48.958333,49.828789,50.043113,"""Papel, celulose e gráfica"""
"""22""",49.673723,50.727993,49.859299,"""Plástico e borracha"""


## 8. Figuras e recomendações candidatas

Os PNGs vão para `reports/figures/` (slides). As recomendações saem como **rascunho** em `app/recomendacoes.yaml`, cada uma com o indicador que mostra se a decisão funcionou e a linha de base de hoje; a equipe revisa e aprova.

In [11]:
figuras.main()
recomendacoes.main()

PNG: reports/figures/01_pedido_por_grupo_setorial.png


PNG: reports/figures/02_t3_com_sem_alimentador.png
Recomendações pendentes: faltam as microrregiões do IBGE (V21) e/ou a classificação das macrofamílias (Fase 2)


[]

## Premissas e limites

Ver a seção "Premissas e limites" do README. A principal: o CAGED mede fluxo, não vagas abertas; "apagão" é inferido por sinais indiretos, e escolaridade e salário são **associados** aos gargalos, sem afirmação de causa.